# Big Data Analysis of Google Play Store Applications / Group 22
###### Team Members: Lotta Kauppinen, Jenna Kiviaho, Marjaana Koski, Jani Laakso, Aleksi Savukoski

#### The chosen dataset contains information of more than 2.3 million applications from Google Play Store. The main goal of this analysis is to find different patterns between the variables. By doing that, we are able to analyze how the different characteristics affect the popularity and ratings.

#### The analysis also aims to showcase if the free apps outperform the paid apps.

#### The other goal for this analysis is to showcase how Spark and MongoDB work together in a Big Data Analysis.


In [92]:
#  Spark Session:

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("GooglePlayStoreAnalysis").getOrCreate()

In [93]:
# Downloading the dataset:
# This is outdated, check next df definition /Marjaana
# df = spark.read.csv("data\Google-Playstore.csv",header=True,inferSchema=True)
# df.printSchema()

In [94]:
# Due to e.g. commas, escape quotes and other special characters, careful CSV parsing is required
df = (spark.read.option("header", "true").option("inferSchema", "true").option("quote", "\"").option("escape", "\"").option("multiLine", "true").csv("data/Google-Playstore.csv"))
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Developer Website: string (nullable = true)
 |-- Developer Email: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Privacy Policy: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)
 |-- Scr

In [95]:
# The amount of rows and columns we are working with:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 2312944
Columns: 24


## Cleaning the data

In [96]:
# Checking the amount of null values by column:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Currency|Size|Minimum Android|Developer Id|Developer Website|Developer Email|Released|Last Updated|Content Rating|Privacy Policy|Ad Supported|In App Purchases|Editors Choice|Scraped Time|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0|     135| 196|           6530|      

### Dropping columns that are not needed

In [97]:
# Due to the purpose of this analysis, we are not going to need the information of the columns: 
# Developer Website, Developer Email, Privacy Policy, Scraped Time, so let's drop them.
# Currency has multiple values, so that should remain as it affects the Price field.
df = df.drop("Developer Website","Developer Email","Privacy Policy","Scraped Time")

### Handling NULL-values

In [98]:
# Let's check the remaining nulls and make a plan to handle them.
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Currency|Size|Minimum Android|Developer Id|Released|Last Updated|Content Rating|Ad Supported|In App Purchases|Editors Choice|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0|     135| 196|           6530|          33|   71053|           0|             0|           0|               0|             0|
+--------+------+--------+------+------------+--------+----------------+----------------+----+--

In [99]:
# This is not needed, field is double / Marjaana

from pyspark.sql.functions import col

# Let's find the rows that include values that ar NOT int or float.
non_numeric = df.filter(~col("Rating").rlike(r'^-?\d+(\.\d+)?$'))

# Examples of the values that are NOT int or float.
non_numeric.select("Rating").show(100, truncate=False)


+------+
|Rating|
+------+
+------+



#### Rating and Rating Count -columns: NULL-values
##### These columns includes the average rating and number of rating of each app. All of the NULL-values and values that are not numeric will be changed to 0.0 (Rating) and 0 (Rating Count).

In [9]:
# This should not be done, value should remain null / Marjaana
# from pyspark.sql.functions import col, when, replace

# # It seems that some of the App-values have been written in the wrong columns. Let's fill all the non-numeric values with 0.0.
# df = df.withColumn(
#     "Rating_num",
#     when(col("Rating").rlike(r'^-?\d+(\.\d+)?$'), col("Rating").cast("float"))
#     .otherwise(0.0)
# )

# # To check that the code worked:
# df.select("Rating", "Rating_num").filter(col("Rating_num") == 0.0).show(100, truncate=False)

In [10]:
# df = df.withColumn(
#     "Rating Count", 
#     when(col("Rating_num") == 0.0, "0")  # Rating_num == 0.0 > Rating Count = 0
#     .otherwise(col("Rating Count"))      
# )

# # To check that the code worked:
# df.select("Rating", "Rating_num", "Rating Count").filter(col("Rating_num") == 0.0).show(100, truncate=False)
# df = df.drop("Rating")

### Changing the data types

In [100]:
# Checking the data types
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)



In [101]:
from pyspark.sql.functions import translate, when, round, expr, lower

# Changing data types
df = df.withColumn("Installs", expr("try_cast(translate(Installs, '+,', '') as long)")) # Removing characters '+' and ','
df = df.withColumn("Price", expr("try_cast(Price as float)"))
# This is already boolean
# df = df.withColumn("Free", 
#     when(lower(col("Free").cast("string")) == "true", True)
#     .when(lower(col("Free").cast("string")) == "false", False)
#     .otherwise(None)
# )

# Handling column "Size"
df = df.withColumn("Size", 
    when(col("Size").contains("k"),
         expr("try_cast(translate(Size, 'k,', ' .') as float)") / 1024) # Removing 'k' and ',', then divided by 1024 so every size is in MB
    .when(col("Size").contains("M"), 
         expr("try_cast(translate(Size, 'M,', ' .') as float)")) # Removing characters 'M' and ','
    .otherwise(expr("try_cast(Size as float)")) # If it's only a number and no characters
)

# Rounding column "Size"
df = df.withColumn("Size", round(col("Size"), 2))

# Checking data types again to make sure they are correct
print(df.schema["Installs"].dataType)
print(df.schema["Price"].dataType)
print(df.schema["Rating Count"].dataType)
print(df.schema["Free"].dataType)
print(df.schema["Size"].dataType)


LongType()
FloatType()
IntegerType()
BooleanType()
DoubleType()


In [102]:
# These are already boolean
# df = df.withColumn("Ad Supported", 
#     when(lower(col("Ad Supported").cast("string")) == "true", True)
#     .when(lower(col("Ad Supported").cast("string")) == "false", False)
#     .otherwise(None)
# )
# df = df.withColumn("Editors Choice", 
#     when(lower(col("Editors Choice").cast("string")) == "true", True)
#     .when(lower(col("Editors Choice").cast("string")) == "false", False)
#     .otherwise(None)
# )
# df = df.withColumn("In App Purchases", 
#     when(lower(col("In App Purchases").cast("string")) == "true", True)
#     .when(lower(col("In App Purchases").cast("string")) == "false", False)
#     .otherwise(None)
# )

df = df.withColumn("Minimum Installs", expr("try_cast(`Minimum Installs` as long)"))
df = df.withColumn("Maximum Installs", expr("try_cast(`Maximum Installs` as long)"))

print(df.schema["Ad Supported"].dataType)
print(df.schema["Editors Choice"].dataType)
print(df.schema["In App Purchases"].dataType)
print(df.schema["Minimum Installs"].dataType)
print(df.schema["Maximum Installs"].dataType)

BooleanType()
BooleanType()
BooleanType()
LongType()
LongType()


In [103]:
from pyspark.sql.functions import try_to_date

# Using try_to_date function so unexpected values will be filled with NULL
df = df.withColumn("Released", try_to_date(col("Released"), "MMM d, yyyy")) \
       .withColumn("Last Updated", try_to_date(col("Last Updated"), "MMM d, yyyy"))

# Checking the outcome
df.select("Released", "Last Updated").show(20)
print(df.schema["Released"].dataType)
print(df.schema["Last Updated"].dataType)


+----------+------------+
|  Released|Last Updated|
+----------+------------+
|2020-02-26|  2020-02-26|
|2020-05-21|  2021-05-06|
|2019-08-09|  2019-08-19|
|2018-09-10|  2018-10-13|
|2020-02-21|  2018-11-12|
|2018-12-24|  2019-12-20|
|2019-09-23|  2019-09-27|
|2019-06-21|  2019-06-21|
|      NULL|  2018-12-07|
|2019-09-22|  2020-10-07|
|2020-07-30|  2020-07-30|
|2018-01-10|  2018-06-27|
|2018-04-03|  2021-06-11|
|2020-02-09|  2021-05-14|
|2018-09-05|  2020-05-30|
|2020-04-05|  2021-03-23|
|2016-11-28|  2019-10-30|
|2019-04-24|  2019-05-05|
|2020-07-01|  2021-05-26|
|2020-12-26|  2021-03-23|
+----------+------------+
only showing top 20 rows
DateType()
DateType()


In [104]:
# Confirm the data types after fixes
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: float (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: double (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: date (nullable = true)
 |-- Last Updated: date (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)



#### Dropping columns with null values

There are more than 1 million rows with rating value as 0. Then again, these rows do not have rating count above 0. Earlier it was noted, that there are approximately 22 883 rows where both rating and rating count are null.

To convert the rows with zero ratings to a single representation, the 0 ratings and counts are converted to nulls. This way the non rated apps remain as null values, as 0 is not a rating that can be given for an app.

In [105]:
print(f'Rows with null ratings and rating count: {df.filter((col("Rating").isNull()) & (col("Rating Count").isNull())).count()}')
print(f'Rows with rating value as 0: {df.filter(col("Rating") == 0).count()}')
print(f'Rows with rating value as 0 and rating count > 0 : {df.filter((col("Rating") == 0) & (col("Rating Count") > 0)).count()}')
print(f'Rows with rating value > 0 and rating count as 0 : {df.filter((col("Rating") > 0) & (col("Rating Count") == 0)).count()}')
print(f'Rows with rating value as not null and rating count as null: {df.filter((col("Rating").isNotNull()) & (col("Rating Count").isNull())).count()}')
print(f'Rows with rating value as null and rating count as not null: {df.filter((col("Rating").isNull()) & (col("Rating Count").isNotNull())).count()}')

Rows with null ratings and rating count: 22883
Rows with rating value as 0: 1059762
Rows with rating value as 0 and rating count > 0 : 0
Rows with rating value > 0 and rating count as 0 : 0
Rows with rating value as not null and rating count as null: 0
Rows with rating value as null and rating count as not null: 0


In [106]:
df = df.withColumn("Rating",when((col("Rating") == 0) & (col("Rating Count") == 0), None).otherwise(col("Rating")))

df = df.withColumn("Rating Count",when((col("Rating Count") == 0) & (col("Rating").isNull()), None).otherwise(col("Rating Count")))

In [107]:
# Now all 0 ratings and rating counts are consistently null
df.filter((col("Rating").isNull()) & (col("Rating Count").isNull())).count()

1082645

We retained Size and Released columns despite their ~6% null rate. Removing them would narrow the data significantly and could have potentially biased the analysis of other attributes like popularity and categories.

As we want to study the popularitity against multiple variables, rows below containing null values are dropped out. Out of these the installations are the most important ones, as application without installations cannot have reviews or meaningful data. The remaining columns contain information we want to clean, and therefore these rows are dropped. This diminishes the dataset only by ~0.29 % while making the data clearer.

In [111]:
df_null= df.na.drop(subset=['Installs'])
df_no_null = df_null.na.drop(subset=['Installs', 'Minimum Installs', 'Ad Supported', 'Developer Id', 'In App Purchases', 'Free', 'Minimum Android'])

print(df.count())
print(df_null.count())
print(df_no_null.count())
print((df.count()-df_no_null.count())/df.count()*100)

2312944
2312837
2306274
0.2883770640361375


In [ ]:
df = df.na.drop(subset=['Installs', 'Minimum Installs', 'Ad Supported', 'Developer Id', 'In App Purchases', 'Free', 'Minimum Android'])
df.count()

2306274

#### Handling Duplicates

Due to the nature of this data, we need to check the rows that have the same App ID. Because there are no such rows, nothing needs to be deleted.

In [ ]:
from pyspark.sql.functions import count
df.groupBy("App Id").count().filter("count > 1").show()

+------+-----+
|App Id|count|
+------+-----+
+------+-----+



## Dependency analysis of apps ratings and price type

In [18]:
from pyspark.sql.functions import col, avg, stddev, count, when, round as spark_round
import pandas as pd
import matplotlib.pyplot as plt

df.createOrReplaceTempView("apps")

# ============================================================
# ANALYSIS 1: BASIC STATISTICS - FREE VS PAID RATINGS
# ============================================================

print("="*60)
print("RATING STATISTICS: FREE VS PAID APPS")
print("="*60)

rating_stats = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps_with_ratings,
        ROUND(AVG(Rating), 2) AS avg_rating,
        ROUND(STDDEV(Rating), 2) AS stddev_rating,
        ROUND(MIN(Rating), 2) AS min_rating,
        ROUND(MAX(Rating), 2) AS max_rating,
        ROUND(PERCENTILE_APPROX(Rating, 0.5), 2) AS median_rating
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free
""")

rating_stats.show(truncate=False)

# ============================================================
# ANALYSIS 2: RATING DISTRIBUTION BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("RATING DISTRIBUTION: FREE VS PAID")
print("="*60)

rating_distribution = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        CASE 
            WHEN Rating >= 4.5 THEN '4.5 - 5.0 (Excellent)'
            WHEN Rating >= 4.0 THEN '4.0 - 4.5 (Good)'
            WHEN Rating >= 3.5 THEN '3.5 - 4.0 (Average)'
            WHEN Rating >= 3.0 THEN '3.0 - 3.5 (Below Average)'
            WHEN Rating >= 2.0 THEN '2.0 - 3.0 (Poor)'
            ELSE 'Below 2.0 (Very Poor)'
        END AS rating_category,
        COUNT(*) AS app_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Free), 2) AS percentage
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free, rating_category
    ORDER BY Free DESC, rating_category
""")

rating_distribution.show(20, truncate=False)

# ============================================================
# ANALYSIS 3: STATISTICAL SIGNIFICANCE
# ============================================================

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TEST")
print("="*60)

significance = spark.sql("""
    WITH stats AS (
        SELECT 
            Free,
            COUNT(*) AS n,
            AVG(Rating) AS mean,
            STDDEV(Rating) AS stddev,
            VARIANCE(Rating) AS variance
        FROM apps
        WHERE Rating IS NOT NULL AND Rating > 0
        GROUP BY Free
    )
    SELECT 
        MAX(CASE WHEN Free = true THEN mean END) AS free_mean,
        MAX(CASE WHEN Free = false THEN mean END) AS paid_mean,
        MAX(CASE WHEN Free = true THEN stddev END) AS free_stddev,
        MAX(CASE WHEN Free = false THEN stddev END) AS paid_stddev,
        MAX(CASE WHEN Free = true THEN n END) AS free_n,
        MAX(CASE WHEN Free = false THEN n END) AS paid_n,
        ABS(MAX(CASE WHEN Free = true THEN mean END) - MAX(CASE WHEN Free = false THEN mean END)) AS mean_difference
    FROM stats
""").collect()[0]

print(f"Free apps average rating: {significance['free_mean']:.2f}")
print(f"Paid apps average rating: {significance['paid_mean']:.2f}")
print(f"Difference: {significance['mean_difference']:.2f}")

# Calculate effect size (Cohen's d approximation)
pooled_std = ((significance['free_stddev']**2 + significance['paid_stddev']**2) / 2) ** 0.5
effect_size = significance['mean_difference'] / pooled_std

print(f"\nEffect size (Cohen's d): {effect_size:.3f}")
if abs(effect_size) < 0.2:
    print("Interpretation: Negligible effect")
elif abs(effect_size) < 0.5:
    print("Interpretation: Small effect")
elif abs(effect_size) < 0.8:
    print("Interpretation: Medium effect")
else:
    print("Interpretation: Large effect")

# ============================================================
# ANALYSIS 4: TOP RATED APPS BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED FREE APPS")
print("="*60)

top_free = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(Rating, 2) AS rating,
        `Rating Count` AS review_count
    FROM apps
    WHERE Free = true 
        AND Rating IS NOT NULL 
        AND Rating > 0
    ORDER BY Rating DESC
    LIMIT 5
""")
top_free.show(truncate=False)

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED PAID APPS")
print("="*60)

top_paid = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(Rating, 2) AS rating,
        `Rating Count` AS review_count,
        CONCAT('$', Price) AS price
    FROM apps
    WHERE Free = false 
        AND Rating IS NOT NULL 
        AND Rating > 0
    ORDER BY Rating DESC
    LIMIT 5
""")
top_paid.show(truncate=False)

# ============================================================
# ANALYSIS 5: SUMMARY AND CONCLUSION
# ============================================================

print("\n" + "="*60)
print("SUMMARY AND CONCLUSIONS")
print("="*60)

summary = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps_with_real_ratings,
        ROUND(AVG(Rating), 2) AS avg_rating,
        ROUND(PERCENTILE_APPROX(Rating, 0.5), 2) AS median_rating,
        ROUND(STDDEV(Rating), 3) AS rating_stddev,
        COUNT(CASE WHEN Rating >= 4.0 THEN 1 END) AS high_rated_apps,
        ROUND(COUNT(CASE WHEN Rating >= 4.0 THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_high_rated
    FROM apps
    WHERE Rating IS NOT NULL AND Rating > 0
    GROUP BY Free
""")

summary.show(truncate=False)

# Statistical conclusion
free_avg = summary.filter(col("price_type") == "Free").select("avg_rating").collect()[0][0]
paid_avg = summary.filter(col("price_type") == "Paid").select("avg_rating").collect()[0][0]

print(f"\nKEY FINDINGS (REAL RATINGS ONLY):")
print(f"• Free apps average rating: {free_avg:.2f}")
print(f"• Paid apps average rating: {paid_avg:.2f}")
print(f"• Difference: {abs(free_avg - paid_avg):.2f} points")
print(f"• Effect size (Cohen's d): {effect_size:.3f}")

if effect_size < 0.2:
    print("\n• CONCLUSION: NO meaningful dependency between rating and price type")
    print("  The rating difference is negligible - less than 0.2 standard deviations")
elif effect_size < 0.5:
    print("\n• CONCLUSION: Weak dependency - Paid apps tend to have slightly higher ratings")
elif effect_size < 0.8:
    print("\n• CONCLUSION: Moderate dependency - Paid apps have noticeably higher ratings")
else:
    print("\n• CONCLUSION: Strong dependency - Paid apps have substantially higher ratings")

RATING STATISTICS: FREE VS PAID APPS
+----------+-----------------------+----------+-------------+----------+----------+-------------+
|price_type|total_apps_with_ratings|avg_rating|stddev_rating|min_rating|max_rating|median_rating|
+----------+-----------------------+----------+-------------+----------+----------+-------------+
|Free      |1202394                |4.1       |0.69         |1.0       |5.0       |4.2          |
|Paid      |25396                  |4.15      |0.62         |1.0       |5.0       |4.3          |
+----------+-----------------------+----------+-------------+----------+----------+-------------+


RATING DISTRIBUTION: FREE VS PAID
+----------+-------------------------+---------+----------+
|price_type|rating_category          |app_count|percentage|
+----------+-------------------------+---------+----------+
|Free      |2.0 - 3.0 (Poor)         |71698    |5.96      |
|Free      |3.0 - 3.5 (Below Average)|103777   |8.63      |
|Free      |3.5 - 4.0 (Average)      |2